In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

# Project root detect karo
ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

# Cleaned Bitext dataset path
DATA_PATH = (
    ROOT
    / "data"
    / "processed"
    / "bitext"
    / "bitext_customer_support_clean.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset nahi mili: {DATA_PATH}")

# Dataset load karo
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("-" * 50)
print("Project root:", ROOT)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

display(df.head(5))

Dataset loaded successfully!
--------------------------------------------------
Project root: d:\Data scientist\AI Customer Support Intelligence
Dataset shape: (26872, 5)
Columns: ['flags', 'instruction', 'category', 'intent', 'response']

Data types:
flags          str
instruction    str
category       str
intent         str
response       str
dtype: object

Missing values:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [2]:
# 1. Dataset overview
print("Total records:", f"{len(df):,}")
print("Total categories:", df["category"].nunique())
print("Total intents:", df["intent"].nunique())

# 2. Category distribution
print("\n--- Category Distribution ---")
category_counts = df["category"].value_counts()

display(
    category_counts.rename_axis("category")
    .reset_index(name="count")
)

# 3. Top 15 most frequent intents
print("\n--- Top 15 Intents ---")
intent_counts = df["intent"].value_counts()

display(
    intent_counts.head(15)
    .rename_axis("intent")
    .reset_index(name="count")
)

# 4. Check duplicate records
print("\n--- Duplicate Analysis ---")
print("Completely duplicate rows:", df.duplicated().sum())

print(
    "Duplicate instructions:",
    df["instruction"].duplicated().sum()
)

# 5. Check text lengths
print("\n--- Text Length Statistics ---")

df["instruction_length"] = df["instruction"].astype(str).str.len()
df["response_length"] = df["response"].astype(str).str.len()

display(
    df[["instruction_length", "response_length"]].describe().round(2)
)

Total records: 26,872
Total categories: 11
Total intents: 27

--- Category Distribution ---


,category,count
0,ACCOUNT,5986
1,ORDER,3988
2,REFUND,2992
3,INVOICE,1999
4,CONTACT,1999
5,PAYMENT,1998
6,FEEDBACK,1997
7,DELIVERY,1994
8,SHIPPING,1970
9,SUBSCRIPTION,999



--- Top 15 Intents ---


,intent,count
0,check_invoice,1000
1,complaint,1000
2,contact_customer_service,1000
3,edit_account,1000
4,switch_account,1000
5,check_payment_methods,999
6,contact_human_agent,999
7,delivery_period,999
8,get_invoice,999
9,newsletter_subscription,999



--- Duplicate Analysis ---
Completely duplicate rows: 0
Duplicate instructions: 2237

--- Text Length Statistics ---


,instruction_length,response_length
count,26872.00,26872.00
mean,46.89,634.10
std,10.90,331.59
min,6.00,57.00
25%,40.00,427.00
50%,48.00,540.00
75%,55.00,753.00
max,92.00,2472.00


In [3]:
# Normalize queries for duplicate analysis only.
# Original dataset is not modified.
df["instruction_normalized"] = (
    df["instruction"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Group repeated customer instructions
query_groups = (
    df.groupby("instruction_normalized")
    .agg(
        occurrences=("instruction_normalized", "size"),
        unique_intents=("intent", "nunique"),
        unique_categories=("category", "nunique"),
        intents=("intent", lambda x: ", ".join(sorted(x.unique()))),
        categories=("category", lambda x: ", ".join(sorted(x.unique()))),
    )
    .reset_index()
)

duplicate_groups = query_groups[
    query_groups["occurrences"] > 1
].sort_values("occurrences", ascending=False)

conflicting_intents = duplicate_groups[
    duplicate_groups["unique_intents"] > 1
]

conflicting_categories = duplicate_groups[
    duplicate_groups["unique_categories"] > 1
]

print("Unique normalized queries:", len(query_groups))
print("Repeated query groups:", len(duplicate_groups))
print("Repeated groups with conflicting intents:", len(conflicting_intents))
print("Repeated groups with conflicting categories:", len(conflicting_categories))

print("\n--- Most frequently repeated queries ---")
display(duplicate_groups.head(10))

print("\n--- Queries mapped to multiple intents ---")
display(conflicting_intents.head(15))

print("\n--- Queries mapped to multiple categories ---")
display(conflicting_categories.head(15))

Unique normalized queries: 24371
Repeated query groups: 1144
Repeated groups with conflicting intents: 0
Repeated groups with conflicting categories: 0

--- Most frequently repeated queries ---


,instruction_normalized,occurrences,unique_intents,unique_categories,intents,categories
10909,i expect a rebate of {{refund amount}} dollars,10,1,1,track_refund,REFUND
10911,i expect a refund of {{currency symbol}}{{refu...,10,1,1,track_refund,REFUND
10905,i expect a rebate of {{currency symbol}}{{refu...,10,1,1,track_refund,REFUND
10930,i expect a restitution of {{refund amount}} do...,10,1,1,track_refund,REFUND
10889,i expect a compensation of {{currency symbol}}...,10,1,1,track_refund,REFUND
10892,i expect a compensation of {{refund amount}} d...,10,1,1,track_refund,REFUND
10920,i expect a reimbursement of {{refund amount}} ...,10,1,1,track_refund,REFUND
10923,i expect a restitution of {{currency symbol}}{...,10,1,1,track_refund,REFUND
10916,i expect a refund of {{refund amount}} dollars,10,1,1,track_refund,REFUND
10919,i expect a reimbursement of {{currency symbol}...,9,1,1,track_refund,REFUND



--- Queries mapped to multiple intents ---


,instruction_normalized,occurrences,unique_intents,unique_categories,intents,categories



--- Queries mapped to multiple categories ---


,instruction_normalized,occurrences,unique_intents,unique_categories,intents,categories


In [4]:
from sklearn.model_selection import train_test_split

# Confirm each normalized query has only one intent and category
intent_counts_per_query = (
    df.groupby("instruction_normalized")["intent"].nunique()
)

category_counts_per_query = (
    df.groupby("instruction_normalized")["category"].nunique()
)

assert intent_counts_per_query.max() == 1, (
    "Some repeated queries have conflicting intents."
)
assert category_counts_per_query.max() == 1, (
    "Some repeated queries have conflicting categories."
)

# Create one record per unique normalized query for splitting
unique_queries = (
    df.groupby("instruction_normalized", as_index=False)
    .agg(
        intent=("intent", "first"),
        category=("category", "first"),
    )
)

# Split query groups, stratified by intent
train_queries, val_queries = train_test_split(
    unique_queries,
    test_size=0.20,
    random_state=42,
    stratify=unique_queries["intent"],
)

train_query_set = set(train_queries["instruction_normalized"])
val_query_set = set(val_queries["instruction_normalized"])

# Include all original rows for each assigned query group
train_df = df[
    df["instruction_normalized"].isin(train_query_set)
].copy()

val_df = df[
    df["instruction_normalized"].isin(val_query_set)
].copy()

# Verify that no normalized query occurs in both splits
overlap = (
    set(train_df["instruction_normalized"])
    & set(val_df["instruction_normalized"])
)

assert len(overlap) == 0, "Data leakage detected!"

print("Train rows:", f"{len(train_df):,}")
print("Validation rows:", f"{len(val_df):,}")
print("Training unique queries:", train_df["instruction_normalized"].nunique())
print("Validation unique queries:", val_df["instruction_normalized"].nunique())
print("Query overlap:", len(overlap))

print("\nTraining intent distribution:")
display(train_df["intent"].value_counts().sort_index())

print("\nValidation intent distribution:")
display(val_df["intent"].value_counts().sort_index())

Train rows: 21,525
Validation rows: 5,347
Training unique queries: 19496
Validation unique queries: 4875
Query overlap: 0

Training intent distribution:


intent
cancel_order                790
change_order                799
change_shipping_address     780
check_cancellation_fee      758
check_invoice               803
check_payment_methods       799
check_refund_policy         800
complaint                   800
contact_customer_service    799
contact_human_agent         797
create_account              795
delete_account              798
delivery_options            828
delivery_period             801
edit_account                816
get_invoice                 792
get_refund                  786
newsletter_subscription     799
payment_issue               800
place_order                 799
recover_password            797
registration_problems       799
review                      797
set_up_shipping_address     797
switch_account              797
track_order                 799
track_refund                800
Name: count, dtype: int64


Validation intent distribution:


intent
cancel_order                208
change_order                198
change_shipping_address     193
check_cancellation_fee      192
check_invoice               197
check_payment_methods       200
check_refund_policy         197
complaint                   200
contact_customer_service    201
contact_human_agent         202
create_account              202
delete_account              197
delivery_options            167
delivery_period             198
edit_account                184
get_invoice                 207
get_refund                  211
newsletter_subscription     200
payment_issue               199
place_order                 199
recover_password            198
registration_problems       200
review                      200
set_up_shipping_address     200
switch_account              203
track_order                 196
track_refund                198
Name: count, dtype: int64

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Input text and target labels
X_train_text = train_df["instruction"].astype(str)
X_val_text = val_df["instruction"].astype(str)

y_train = train_df["intent"].astype(str)
y_val = val_df["intent"].astype(str)

# Configure TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=40000,
    sublinear_tf=True,
    dtype=np.float32,
)

# Fit ONLY on training data
X_train_vec = vectorizer.fit_transform(X_train_text)

# Transform validation data using the same vocabulary
X_val_vec = vectorizer.transform(X_val_text)

print("TF-IDF feature extraction complete!")
print("-" * 45)

print("Training records:", X_train_vec.shape[0])
print("Validation records:", X_val_vec.shape[0])
print("Vocabulary size:", len(vectorizer.vocabulary_))

print("\nTraining matrix shape:", X_train_vec.shape)
print("Validation matrix shape:", X_val_vec.shape)

print("\nTraining intent classes:", y_train.nunique())
print("Validation intent classes:", y_val.nunique())

print("\nSample vocabulary terms:")
print(list(vectorizer.vocabulary_.keys())[:20])

print("\nSparse matrix density:")
density = (
    X_train_vec.nnz
    / (X_train_vec.shape[0] * X_train_vec.shape[1])
)
print(f"{density:.6%}")

TF-IDF feature extraction complete!
---------------------------------------------
Training records: 21525
Validation records: 5347
Vocabulary size: 5193

Training matrix shape: (21525, 5193)
Validation matrix shape: (5347, 5193)

Training intent classes: 27
Validation intent classes: 27

Sample vocabulary terms:
['question', 'about', 'cancelling', 'order', 'number', 'question about', 'about cancelling', 'cancelling order', 'order order', 'order number', 'have', 'oorder', 'have question', 'oorder order', 'need', 'help', 'puchase', 'need help', 'help cancelling', 'cancelling puchase']

Sparse matrix density:
0.268336%


In [6]:
import time
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

# Define baseline models
models = {
    "Logistic Regression": LogisticRegression(
        C=4.0,
        max_iter=1500,
        solver="lbfgs",
        random_state=42,
    ),

    "Linear SVM": LinearSVC(
        C=1.5,
        max_iter=3000,
        random_state=42,
    ),

    "Multinomial Naive Bayes": MultinomialNB(
        alpha=0.3,
    ),
}

trained_models = {}
results = []

# Train and evaluate each model on the validation set
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    start_time = time.time()

    model.fit(X_train_vec, y_train)
    predictions = model.predict(X_val_vec)

    elapsed = time.time() - start_time

    accuracy = accuracy_score(y_val, predictions)
    macro_f1 = f1_score(
        y_val, predictions, average="macro", zero_division=0
    )
    weighted_f1 = f1_score(
        y_val, predictions, average="weighted", zero_division=0
    )

    trained_models[model_name] = model

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "Time (seconds)": round(elapsed, 2),
    })

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print(f"Training + prediction time: {elapsed:.2f} seconds")

# Compare results
results_df = (
    pd.DataFrame(results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

print("\n--- Validation Model Comparison ---")
display(results_df.round(4))

# Select the leading baseline by Macro F1
best_baseline_name = results_df.iloc[0]["Model"]

print("Current leading baseline:", best_baseline_name)
print("Selection metric: Validation Macro F1")


Training Logistic Regression...
Accuracy: 0.9950
Macro F1: 0.9950
Weighted F1: 0.9950
Training + prediction time: 4.35 seconds

Training Linear SVM...
Accuracy: 0.9948
Macro F1: 0.9948
Weighted F1: 0.9948
Training + prediction time: 1.41 seconds

Training Multinomial Naive Bayes...
Accuracy: 0.9875
Macro F1: 0.9876
Weighted F1: 0.9875
Training + prediction time: 0.12 seconds

--- Validation Model Comparison ---


,Model,Accuracy,Macro F1,Weighted F1,Time (seconds)
0,Logistic Regression,0.9950,0.9950,0.9950,4.35
1,Linear SVM,0.9948,0.9948,0.9948,1.41
2,Multinomial Naive Bayes,0.9875,0.9876,0.9875,0.12


Current leading baseline: Logistic Regression
Selection metric: Validation Macro F1


In [7]:
import time
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# Random Forest configuration
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=30,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=2,
    random_state=42,
)

print("Training Random Forest...")
start_time = time.time()

rf_model.fit(X_train_vec, y_train)

rf_predictions = rf_model.predict(X_val_vec)
rf_time = time.time() - start_time

# Validation metrics
rf_accuracy = accuracy_score(y_val, rf_predictions)

rf_macro_f1 = f1_score(
    y_val,
    rf_predictions,
    average="macro",
    zero_division=0,
)

rf_weighted_f1 = f1_score(
    y_val,
    rf_predictions,
    average="weighted",
    zero_division=0,
)

print("\n--- Random Forest Results ---")
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"Macro F1: {rf_macro_f1:.4f}")
print(f"Weighted F1: {rf_weighted_f1:.4f}")
print(f"Training + prediction time: {rf_time:.2f} seconds")

# Preserve model and add its result to existing comparison
trained_models["Random Forest"] = rf_model

results.append({
    "Model": "Random Forest",
    "Accuracy": rf_accuracy,
    "Macro F1": rf_macro_f1,
    "Weighted F1": rf_weighted_f1,
    "Time (seconds)": round(rf_time, 2),
})

results_df = (
    pd.DataFrame(results)
    .drop_duplicates(subset=["Model"], keep="last")
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

print("\n--- Updated Model Comparison ---")
display(results_df.round(4))

Training Random Forest...

--- Random Forest Results ---
Accuracy: 0.9575
Macro F1: 0.9577
Weighted F1: 0.9576
Training + prediction time: 2.22 seconds

--- Updated Model Comparison ---


,Model,Accuracy,Macro F1,Weighted F1,Time (seconds)
0,Logistic Regression,0.9950,0.9950,0.9950,4.35
1,Linear SVM,0.9948,0.9948,0.9948,1.41
2,Multinomial Naive Bayes,0.9875,0.9876,0.9875,0.12
3,Random Forest,0.9575,0.9577,0.9576,2.22


In [8]:
import time
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

try:
    from xgboost import XGBClassifier
    xgboost_available = True
except ImportError:
    xgboost_available = False

if not xgboost_available:
    print("XGBoost is not installed.")
    print("Run this in a separate notebook cell:")
    print("%pip install xgboost")

else:
    print("XGBoost is available.")
    print("Training XGBoost model...")

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)

    xgb_model = XGBClassifier(
        n_estimators=80,
        max_depth=4,
        learning_rate=0.15,
        subsample=0.85,
        colsample_bytree=0.65,
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        max_bin=128,
        n_jobs=2,
        random_state=42,
        verbosity=0,
    )

    start_time = time.time()

    xgb_model.fit(X_train_vec, y_train_encoded)

    xgb_predictions_encoded = xgb_model.predict(X_val_vec).astype(int)
    xgb_predictions = label_encoder.inverse_transform(
        xgb_predictions_encoded
    )

    xgb_time = time.time() - start_time

    xgb_accuracy = accuracy_score(y_val, xgb_predictions)
    xgb_macro_f1 = f1_score(
        y_val, xgb_predictions, average="macro", zero_division=0
    )
    xgb_weighted_f1 = f1_score(
        y_val, xgb_predictions, average="weighted", zero_division=0
    )

    trained_models["XGBoost"] = xgb_model

    results.append({
        "Model": "XGBoost",
        "Accuracy": xgb_accuracy,
        "Macro F1": xgb_macro_f1,
        "Weighted F1": xgb_weighted_f1,
        "Time (seconds)": round(xgb_time, 2),
    })

    results_df = (
        pd.DataFrame(results)
        .drop_duplicates(subset=["Model"], keep="last")
        .sort_values("Macro F1", ascending=False)
        .reset_index(drop=True)
    )

    print("\n--- XGBoost Validation Results ---")
    print(f"Accuracy: {xgb_accuracy:.4f}")
    print(f"Macro F1: {xgb_macro_f1:.4f}")
    print(f"Weighted F1: {xgb_weighted_f1:.4f}")
    print(f"Training time: {xgb_time:.2f} seconds")

    print("\n--- All Model Comparison ---")
    display(results_df.round(4))

XGBoost is available.
Training XGBoost model...

--- XGBoost Validation Results ---
Accuracy: 0.9850
Macro F1: 0.9852
Weighted F1: 0.9851
Training time: 54.09 seconds

--- All Model Comparison ---


,Model,Accuracy,Macro F1,Weighted F1,Time (seconds)
0,Logistic Regression,0.9950,0.9950,0.9950,4.35
1,Linear SVM,0.9948,0.9948,0.9948,1.41
2,Multinomial Naive Bayes,0.9875,0.9876,0.9875,0.12
3,XGBoost,0.9850,0.9852,0.9851,54.09
4,Random Forest,0.9575,0.9577,0.9576,2.22


In [9]:
from pathlib import Path
import pandas as pd
import json

from sklearn.metrics import classification_report, confusion_matrix

# Logistic Regression was the leading model on validation data
best_model_name = "Logistic Regression"
best_model = trained_models[best_model_name]

# Validation predictions
val_predictions = best_model.predict(X_val_vec)

# Report output folder
REPORT_DIR = ROOT / "reports" / "models"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Per-intent performance report
report = classification_report(
    y_val,
    val_predictions,
    labels=sorted(y_train.unique()),
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T

print("Per-intent classification report:")
display(
    report_df
    .loc[sorted(y_train.unique()), ["precision", "recall", "f1-score", "support"]]
    .sort_values("f1-score")
    .round(4)
)

# Save detailed report
report_df.to_csv(
    REPORT_DIR / "bitext_logistic_regression_classification_report.csv"
)

with open(
    REPORT_DIR / "bitext_logistic_regression_classification_report.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(report, f, indent=2)

# Inspect incorrectly classified examples
error_df = pd.DataFrame({
    "instruction": X_val_text.reset_index(drop=True),
    "actual_intent": y_val.reset_index(drop=True),
    "predicted_intent": pd.Series(val_predictions),
})

error_df = error_df[
    error_df["actual_intent"] != error_df["predicted_intent"]
].copy()

error_df.to_csv(
    REPORT_DIR / "bitext_misclassified_examples.csv",
    index=False,
)

print("\nIncorrect predictions:", len(error_df))
print("Total validation rows:", len(y_val))
print("\nSample errors:")
display(error_df.head(20))

print("\nReports saved to:", REPORT_DIR)

Per-intent classification report:


,precision,recall,f1-score,support
change_order,0.9703,0.9899,0.9800,198.0
get_refund,0.9952,0.9763,0.9856,211.0
delete_account,0.9800,0.9949,0.9874,197.0
track_refund,0.9754,1.0000,0.9875,198.0
check_invoice,0.9849,0.9949,0.9899,197.0
create_account,0.9950,0.9851,0.9900,202.0
cancel_order,1.0000,0.9808,0.9903,208.0
get_invoice,0.9951,0.9855,0.9903,207.0
registration_problems,0.9900,0.9950,0.9925,200.0
track_order,1.0000,0.9898,0.9949,196.0



Incorrect predictions: 27
Total validation rows: 5347

Sample errors:


,instruction,actual_intent,predicted_intent
13,i bought some damn item help to cancep order {...,cancel_order,change_order
24,how could icancel purchase {{Order Number}},cancel_order,change_order
89,is it possible to cxancel order {{Order Number}},cancel_order,change_order
168,is it possible to cxancel order {{Order Number}},cancel_order,change_order
246,update purchase00123842,change_order,edit_account
356,hep changing some products of an order,change_order,place_order
915,I don't know how I cansee my invoices from {{P...,check_invoice,get_invoice
1371,am I entitled to a reimbursement?,check_refund_policy,get_refund
1993,I want assistance creatinga standard account,create_account,delete_account
2084,what do i have to do to createa premium account,create_account,delete_account



Reports saved to: d:\Data scientist\AI Customer Support Intelligence\reports\models


In [10]:
# Analyze which intent pairs are most frequently confused

import pandas as pd

if error_df.empty:
    print("No misclassified validation examples found.")

else:
    # Count actual-intent -> predicted-intent errors
    confusion_pairs = (
        error_df
        .groupby(
            ["actual_intent", "predicted_intent"],
            as_index=False
        )
        .size()
        .rename(columns={"size": "error_count"})
        .sort_values("error_count", ascending=False)
        .reset_index(drop=True)
    )

    print("Total incorrect predictions:", len(error_df))
    print("Distinct confusion pairs:", len(confusion_pairs))

    print("\n--- Most Confused Intent Pairs ---")
    display(confusion_pairs.head(15))

    print("\n--- Error Count by Actual Intent ---")
    errors_by_actual = (
        error_df["actual_intent"]
        .value_counts()
        .rename_axis("actual_intent")
        .reset_index(name="error_count")
    )

    display(errors_by_actual)

    # Save analysis reports
    confusion_pairs.to_csv(
        REPORT_DIR / "bitext_confusion_pairs.csv",
        index=False
    )

    errors_by_actual.to_csv(
        REPORT_DIR / "bitext_errors_by_intent.csv",
        index=False
    )

    print("\nConfusion reports saved to:", REPORT_DIR)

Total incorrect predictions: 27
Distinct confusion pairs: 15

--- Most Confused Intent Pairs ---


,actual_intent,predicted_intent,error_count
0,get_refund,track_refund,5
1,cancel_order,change_order,4
2,create_account,delete_account,3
3,get_invoice,check_invoice,3
4,track_order,change_order,2
5,change_order,place_order,1
6,change_order,edit_account,1
7,delete_account,switch_account,1
8,check_refund_policy,get_refund,1
9,check_invoice,get_invoice,1



--- Error Count by Actual Intent ---


,actual_intent,error_count
0,get_refund,5
1,cancel_order,4
2,create_account,3
3,get_invoice,3
4,change_order,2
5,payment_issue,2
6,track_order,2
7,check_invoice,1
8,check_refund_policy,1
9,delete_account,1



Confusion reports saved to: d:\Data scientist\AI Customer Support Intelligence\reports\models


In [11]:
import json
import joblib
from pathlib import Path
from datetime import datetime, timezone

# Output folders
MODEL_DIR = ROOT / "models" / "intent_classification"
REPORT_DIR = ROOT / "reports" / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Select the best validation model
best_model_name = "Logistic Regression"
best_model = trained_models[best_model_name]

# Intent-to-category mapping
intent_to_category = (
    df.groupby("intent")["category"]
    .agg(lambda values: sorted(set(values)))
    .to_dict()
)

# Store model and vectorizer together for inference
artifact = {
    "model": best_model,
    "vectorizer": vectorizer,
    "model_name": best_model_name,
    "classes": best_model.classes_.tolist(),
    "intent_to_category": intent_to_category,
    "input_column": "instruction",
    "target_column": "intent",
    "selection_metric": "validation_macro_f1",
    "trained_on_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "query_overlap": 0,
}

model_path = MODEL_DIR / "bitext_intent_model.joblib"

joblib.dump(artifact, model_path, compress=3)

# Save comparison table
results_df = (
    pd.DataFrame(results)
    .drop_duplicates(subset=["Model"], keep="last")
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

results_df.to_csv(
    REPORT_DIR / "bitext_model_comparison.csv",
    index=False,
)

# Save the validation metrics
metrics = {
    "model_name": best_model_name,
    "validation_accuracy": float(
        accuracy_score(y_val, best_model.predict(X_val_vec))
    ),
    "validation_macro_f1": float(
        f1_score(
            y_val,
            best_model.predict(X_val_vec),
            average="macro",
            zero_division=0,
        )
    ),
    "validation_weighted_f1": float(
        f1_score(
            y_val,
            best_model.predict(X_val_vec),
            average="weighted",
            zero_division=0,
        )
    ),
    "validation_rows": int(len(val_df)),
    "training_rows": int(len(train_df)),
    "unique_query_overlap": 0,
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "note": (
        "Validation results are estimates on this dataset. "
        "Real-world generalization has not yet been established."
    ),
}

metrics_path = REPORT_DIR / "bitext_best_model_metrics.json"

with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Model saved successfully!")
print("Model path:", model_path)
print("Comparison report:", REPORT_DIR / "bitext_model_comparison.csv")
print("Metrics report:", metrics_path)
print("\nValidation metrics:")
print(json.dumps(metrics, indent=2))

Model saved successfully!
Model path: d:\Data scientist\AI Customer Support Intelligence\models\intent_classification\bitext_intent_model.joblib
Comparison report: d:\Data scientist\AI Customer Support Intelligence\reports\models\bitext_model_comparison.csv
Metrics report: d:\Data scientist\AI Customer Support Intelligence\reports\models\bitext_best_model_metrics.json

Validation metrics:
{
  "model_name": "Logistic Regression",
  "validation_accuracy": 0.9949504394987844,
  "validation_macro_f1": 0.9950260984305257,
  "validation_weighted_f1": 0.9949555376234567,
  "validation_rows": 5347,
  "training_rows": 21525,
  "unique_query_overlap": 0,
  "saved_at_utc": "2026-10-24T15:13:09.692052+00:00",
  "note": "Validation results are estimates on this dataset. Real-world generalization has not yet been established."
}


In [12]:
import joblib
from pathlib import Path

# Load saved model artifact
model_path = (
    ROOT
    / "models"
    / "intent_classification"
    / "bitext_intent_model.joblib"
)

if not model_path.exists():
    raise FileNotFoundError(f"Saved model not found: {model_path}")

loaded_artifact = joblib.load(model_path)

loaded_model = loaded_artifact["model"]
loaded_vectorizer = loaded_artifact["vectorizer"]
intent_to_category = loaded_artifact["intent_to_category"]

print("Saved model loaded successfully!")
print("Model:", loaded_artifact["model_name"])


def predict_support_intent(message):
    """Predict intent and category for a customer message."""

    if not isinstance(message, str) or not message.strip():
        raise ValueError("Customer message cannot be empty.")

    features = loaded_vectorizer.transform([message.strip()])

    predicted_intent = loaded_model.predict(features)[0]

    # Logistic Regression supports prediction probabilities
    probabilities = loaded_model.predict_proba(features)[0]
    confidence = float(probabilities.max())

    categories = intent_to_category.get(predicted_intent, [])

    return {
        "customer_message": message,
        "predicted_intent": predicted_intent,
        "category": categories[0] if len(categories) == 1 else categories,
        "model_confidence": round(confidence, 4),
    }


# Try some new customer messages
test_messages = [
    "I want to cancel my order",
    "My refund has not arrived yet",
    "I forgot my password and cannot log in",
    "Can you send me a copy of my invoice?",
    "I want to change my shipping address",
]

for message in test_messages:
    result = predict_support_intent(message)

    print("\nCustomer:", result["customer_message"])
    print("Predicted intent:", result["predicted_intent"])
    print("Category:", result["category"])
    print("Model confidence:", result["model_confidence"])

Saved model loaded successfully!
Model: Logistic Regression

Customer: I want to cancel my order
Predicted intent: cancel_order
Category: ORDER
Model confidence: 0.6515

Customer: My refund has not arrived yet
Predicted intent: track_refund
Category: REFUND
Model confidence: 0.4503

Customer: I forgot my password and cannot log in
Predicted intent: recover_password
Category: ACCOUNT
Model confidence: 0.7717

Customer: Can you send me a copy of my invoice?
Predicted intent: get_invoice
Category: INVOICE
Model confidence: 0.6949

Customer: I want to change my shipping address
Predicted intent: change_shipping_address
Category: SHIPPING
Model confidence: 0.9865


In [13]:
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

# Use training examples only as the retrieval source
retrieval_df = train_df.reset_index(drop=True).copy()

# X_train_vec rows are aligned with train_df
retrieval_labels = retrieval_df["intent"].to_numpy()

CONFIDENCE_THRESHOLD = 0.50


def retrieve_support_response(message):
    """
    Predict intent and retrieve a similar example response.
    Low-confidence predictions are flagged for human review.
    """

    if not isinstance(message, str) or not message.strip():
        raise ValueError("Customer message cannot be empty.")

    # Predict intent
    query_vector = loaded_vectorizer.transform([message.strip()])
    predicted_intent = loaded_model.predict(query_vector)[0]

    probabilities = loaded_model.predict_proba(query_vector)[0]
    confidence = float(probabilities.max())

    matching_categories = intent_to_category.get(predicted_intent, [])
    category = (
        matching_categories[0]
        if len(matching_categories) == 1
        else matching_categories
    )

    result = {
        "customer_message": message,
        "predicted_intent": predicted_intent,
        "category": category,
        "model_confidence": round(confidence, 4),
        "review_required": confidence < CONFIDENCE_THRESHOLD,
        "retrieved_response": None,
        "similarity_score": None,
    }

    # Avoid suggesting a response when the intent prediction is uncertain
    if confidence < CONFIDENCE_THRESHOLD:
        result["status"] = "Human review recommended"
        return result

    # Only search examples belonging to the predicted intent
    mask = retrieval_labels == predicted_intent

    if not mask.any():
        result["status"] = "No matching response examples found"
        result["review_required"] = True
        return result

    candidate_vectors = X_train_vec[mask]
    candidate_rows = retrieval_df.loc[mask].reset_index(drop=True)

    # Find the most similar customer instruction
    similarities = cosine_similarity(
        query_vector,
        candidate_vectors,
    ).ravel()

    best_position = int(np.argmax(similarities))
    best_row = candidate_rows.iloc[best_position]

    result["retrieved_response"] = best_row["response"]
    result["matched_training_query"] = best_row["instruction"]
    result["similarity_score"] = round(
        float(similarities[best_position]), 4
    )
    result["status"] = "Retrieved example response — review before sending"

    return result


# Test with a few customer messages
messages = [
    "I want to cancel my order",
    "My refund has not arrived yet",
    "I forgot my password and cannot log in",
]

for message in messages:
    result = retrieve_support_response(message)

    print("\n" + "=" * 70)
    print("Customer:", result["customer_message"])
    print("Intent:", result["predicted_intent"])
    print("Category:", result["category"])
    print("Model confidence:", result["model_confidence"])
    print("Status:", result["status"])

    if result["retrieved_response"]:
        print("Matched query:", result["matched_training_query"])
        print("Similarity:", result["similarity_score"])
        print("Retrieved response:", result["retrieved_response"])


Customer: I want to cancel my order
Intent: cancel_order
Category: ORDER
Model confidence: 0.6515
Status: Retrieved example response — review before sending
Matched query: I want to cancel purchase {{Order Number}}
Similarity: 0.4981
Retrieved response: I can see that you want to cancel purchase {{Order Number}}. Please accept my sincere apologies for any inconvenience caused. To assist you with the cancellation process, please follow the steps below:

1. Sign in to your account: Access our platform by logging in to the {{Online Company Portal Info}}.
2. Locate your order: Once logged in, navigate to the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.
3. Find the specific purchase: Look for the purchase associated with the order number {{Order Number}} and click on it.
4. Initiate cancellation: On the purchase page, there should be an option labeled '{{Online Order Interaction}}'. Please select this option.
5. Confirm cancellation: Follow any additional promp

In [14]:
import re
import pandas as pd

MIN_CONFIDENCE = 0.55
MIN_SIMILARITY = 0.50

PLACEHOLDER_PATTERN = re.compile(r"\{\{[^{}]+\}\}")


def safe_retrieve_support_response(message):
    result = retrieve_support_response(message)
    review_reasons = []

    confidence = result.get("model_confidence", 0.0)
    similarity = result.get("similarity_score")
    response = result.get("retrieved_response")

    # Check intent prediction confidence
    if confidence < MIN_CONFIDENCE:
        review_reasons.append("Low intent confidence")

    # Check similarity with the retrieved training query
    if similarity is None:
        review_reasons.append("No response retrieved")
    elif similarity < MIN_SIMILARITY:
        review_reasons.append("Low query similarity")

    # Identify unresolved template placeholders
    placeholders = (
        sorted(set(PLACEHOLDER_PATTERN.findall(response)))
        if response
        else []
    )

    if placeholders:
        review_reasons.append("Response contains unresolved placeholders")

    # Dataset responses are candidates, not approved live answers.
    result["review_required"] = True
    result["review_reasons"] = review_reasons or [
        "Human approval required before sending"
    ]
    result["unresolved_placeholders"] = placeholders

    # Withhold the candidate if any quality check fails.
    if review_reasons:
        result["retrieved_response"] = None
        result["status"] = "Response withheld — human review required"
    else:
        result["status"] = "Candidate retrieved — human approval required"

    return result


# Test the safety layer
safety_test_messages = [
    "I want to cancel my order",
    "My refund has not arrived yet",
    "I forgot my password and cannot log in",
]

safety_results = []

for message in safety_test_messages:
    result = safe_retrieve_support_response(message)

    safety_results.append({
        "Customer message": result["customer_message"],
        "Predicted intent": result["predicted_intent"],
        "Confidence": result["model_confidence"],
        "Similarity": result["similarity_score"],
        "Status": result["status"],
        "Review reasons": "; ".join(result["review_reasons"]),
        "Placeholders found": len(result["unresolved_placeholders"]),
        "Response released": result["retrieved_response"] is not None,
    })

display(pd.DataFrame(safety_results))

,Customer message,Predicted intent,Confidence,Similarity,Status,Review reasons,Placeholders found,Response released
0,I want to cancel my order,cancel_order,0.6515,0.4981,Response withheld — human review required,Low query similarity; Response contains unreso...,6,False
1,My refund has not arrived yet,track_refund,0.4503,NaN,Response withheld — human review required,Low intent confidence; No response retrieved,0,False
2,I forgot my password and cannot log in,recover_password,0.7717,0.4768,Response withheld — human review required,Low query similarity; Response contains unreso...,2,False


In [15]:
import re
import json
import pandas as pd
from collections import Counter

# Find all template placeholders such as {{Order Number}}
placeholder_pattern = re.compile(r"\{\{([^{}]+)\}\}")

response_audit = df[["category", "intent", "response"]].copy()

response_audit["placeholders"] = response_audit["response"].apply(
    lambda text: placeholder_pattern.findall(str(text))
)

response_audit["placeholder_count"] = (
    response_audit["placeholders"].apply(len)
)

response_audit["has_placeholders"] = (
    response_audit["placeholder_count"] > 0
)

# Overall statistics
total_rows = len(response_audit)
rows_with_placeholders = int(response_audit["has_placeholders"].sum())

all_placeholders = Counter(
    placeholder.strip()
    for placeholders in response_audit["placeholders"]
    for placeholder in placeholders
)

print("Total responses:", total_rows)
print("Responses containing placeholders:", rows_with_placeholders)
print(
    "Percentage containing placeholders:",
    round(100 * rows_with_placeholders / total_rows, 2),
)
print("Unique placeholder types:", len(all_placeholders))

print("\n--- Most common placeholders ---")

placeholder_df = pd.DataFrame(
    all_placeholders.most_common(),
    columns=["placeholder", "occurrences"],
)

display(placeholder_df.head(25))

print("\n--- Placeholder usage by intent ---")

intent_audit = (
    response_audit.groupby("intent")
    .agg(
        total_responses=("response", "size"),
        responses_with_placeholders=("has_placeholders", "sum"),
        average_placeholders=("placeholder_count", "mean"),
    )
    .reset_index()
    .sort_values("responses_with_placeholders", ascending=False)
)

intent_audit["average_placeholders"] = (
    intent_audit["average_placeholders"].round(2)
)

display(intent_audit.head(15))

# Save audit reports
REPORT_DIR.mkdir(parents=True, exist_ok=True)

placeholder_df.to_csv(
    REPORT_DIR / "bitext_placeholder_frequency.csv",
    index=False,
)

intent_audit.to_csv(
    REPORT_DIR / "bitext_placeholder_audit_by_intent.csv",
    index=False,
)

placeholder_summary = {
    "total_responses": total_rows,
    "responses_with_placeholders": rows_with_placeholders,
    "percentage_with_placeholders": round(
        100 * rows_with_placeholders / total_rows, 2
    ),
    "unique_placeholder_types": len(all_placeholders),
    "placeholder_occurrences": dict(all_placeholders),
}

with open(
    REPORT_DIR / "bitext_placeholder_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(placeholder_summary, f, indent=2, ensure_ascii=False)

print("\nPlaceholder audit saved to:", REPORT_DIR)

Total responses: 26872
Responses containing placeholders: 13006
Percentage containing placeholders: 48.4
Unique placeholder types: 385

--- Most common placeholders ---


,placeholder,occurrences
0,Order Number,5122
1,Account Type,4436
2,Account Category,3080
3,Online Order Interaction,2699
4,Customer Support Phone Number,2635
5,Website URL,2534
6,Customer Support Hours,2325
7,Invoice Number,1513
8,Online Company Portal Info,1055
9,Tracking Number,896



--- Placeholder usage by intent ---


,intent,total_responses,responses_with_placeholders,average_placeholders
0,cancel_order,998,995,8.09
25,track_order,995,982,1.64
1,change_order,997,980,1.19
4,check_invoice,1000,976,1.90
15,get_invoice,999,969,1.80
24,switch_account,1000,966,6.16
13,delivery_period,999,941,1.90
12,delivery_options,995,851,2.46
8,contact_customer_service,1000,850,1.46
10,create_account,997,743,1.51



Placeholder audit saved to: d:\Data scientist\AI Customer Support Intelligence\reports\models


In [16]:
import re
import pandas as pd
from pathlib import Path

# Extract all placeholder types and their frequency
placeholder_pattern = re.compile(r"\{\{([^{}]+)\}\}")

placeholder_counts = {}

for response in df["response"].astype(str):
    for match in placeholder_pattern.findall(response):
        name = match.strip()
        placeholder_counts[name] = placeholder_counts.get(name, 0) + 1


def classify_placeholder(name):
    """Suggest a data source; this is a review aid, not an automatic mapping."""

    value = name.lower()

    # Organization-specific configuration
    org_terms = [
        "website url",
        "login page url",
        "phone number",
        "support hours",
        "portal info",
        "company name",
        "support email",
        "store location",
    ]

    if any(term in value for term in org_terms):
        return (
            "organization_config",
            "Configured company settings",
            "Needs configuration",
        )

    # Dynamic customer, account, order or transaction data
    dynamic_terms = [
        "order number",
        "invoice number",
        "tracking number",
        "refund amount",
        "account number",
        "account type",
        "account category",
        "person name",
        "last name",
        "salutation",
        "delivery city",
        "delivery country",
        "date range",
        "currency symbol",
        "customer id",
    ]

    if any(term in value for term in dynamic_terms):
        return (
            "dynamic_data",
            "Authenticated customer or verified backend",
            "Needs field mapping",
        )

    # Labels and UI names may differ across organizations
    ui_terms = [
        "settings",
        "profile",
        "online order interaction",
        "account change",
        "upgrade account",
        "forgot password",
    ]

    if any(term in value for term in ui_terms):
        return (
            "ui_or_workflow_config",
            "Verified application UI/workflow configuration",
            "Needs manual review",
        )

    return (
        "unknown",
        "Determine the authoritative source",
        "Needs manual review",
    )


registry_rows = []

for placeholder, occurrences in sorted(
    placeholder_counts.items(),
    key=lambda item: item[1],
    reverse=True,
):
    field_type, required_source, status = classify_placeholder(placeholder)

    registry_rows.append({
        "placeholder": placeholder,
        "occurrences": occurrences,
        "suggested_type": field_type,
        "required_source": required_source,
        "replacement_value": "",
        "status": status,
    })


placeholder_registry = pd.DataFrame(registry_rows)

REPORT_DIR = ROOT / "reports" / "models"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

registry_path = REPORT_DIR / "bitext_placeholder_registry.csv"

placeholder_registry.to_csv(
    registry_path,
    index=False,
    encoding="utf-8",
)

print("Placeholder registry created!")
print("Unique placeholder types:", len(placeholder_registry))
print("Total placeholder occurrences:", int(
    placeholder_registry["occurrences"].sum()
))

print("\nSuggested classification:")
display(
    placeholder_registry.groupby("suggested_type")
    .agg(
        unique_placeholders=("placeholder", "count"),
        total_occurrences=("occurrences", "sum"),
    )
    .reset_index()
)

print("\nTop 20 placeholders:")
display(placeholder_registry.head(20))

print("\nRegistry saved to:", registry_path)
print("Replacement values are intentionally left blank.")

Placeholder registry created!
Unique placeholder types: 385
Total placeholder occurrences: 35772

Suggested classification:


,suggested_type,unique_placeholders,total_occurrences
0,dynamic_data,23,19799
1,organization_config,16,9499
2,ui_or_workflow_config,28,5161
3,unknown,318,1313



Top 20 placeholders:


,placeholder,occurrences,suggested_type,required_source,replacement_value,status
0,Order Number,5122,dynamic_data,Authenticated customer or verified backend,,Needs field mapping
1,Account Type,4436,dynamic_data,Authenticated customer or verified backend,,Needs field mapping
2,Account Category,3080,dynamic_data,Authenticated customer or verified backend,,Needs field mapping
3,Online Order Interaction,2699,ui_or_workflow_config,Verified application UI/workflow configuration,,Needs manual review
4,Customer Support Phone Number,2635,organization_config,Configured company settings,,Needs configuration
5,Website URL,2534,organization_config,Configured company settings,,Needs configuration
6,Customer Support Hours,2325,organization_config,Configured company settings,,Needs configuration
7,Invoice Number,1513,dynamic_data,Authenticated customer or verified backend,,Needs field mapping
8,Online Company Portal Info,1055,organization_config,Configured company settings,,Needs configuration
9,Tracking Number,896,dynamic_data,Authenticated customer or verified backend,,Needs field mapping



Registry saved to: d:\Data scientist\AI Customer Support Intelligence\reports\models\bitext_placeholder_registry.csv
Replacement values are intentionally left blank.


In [17]:
import re

PLACEHOLDER_PATTERN = re.compile(r"\{\{([^{}]+)\}\}")


def render_response_template(template, context):
    """
    Fill a response template only when every placeholder has a
    non-empty value supplied by an explicit context dictionary.

    This function does not fetch or invent customer/order data.
    A rendered response still requires human approval before sending.
    """

    if not isinstance(template, str) or not template.strip():
        return {
            "status": "blocked",
            "rendered_response": None,
            "missing_fields": [],
            "reason": "Empty or invalid response template",
        }

    if not isinstance(context, dict):
        raise TypeError("context must be a dictionary")

    placeholder_names = [
        name.strip()
        for name in PLACEHOLDER_PATTERN.findall(template)
    ]

    # Normalize keys but preserve the original template.
    normalized_context = {
        str(key).strip().casefold(): value
        for key, value in context.items()
    }

    missing_fields = []

    for name in placeholder_names:
        value = normalized_context.get(name.casefold())

        if value is None or not str(value).strip():
            missing_fields.append(name)

    missing_fields = sorted(set(missing_fields))

    # Do not return a partially filled response.
    if missing_fields:
        return {
            "status": "blocked",
            "rendered_response": None,
            "missing_fields": missing_fields,
            "reason": "Required placeholder values are missing",
        }

    def replace_match(match):
        name = match.group(1).strip()
        value = normalized_context[name.casefold()]
        return str(value).strip()

    rendered = PLACEHOLDER_PATTERN.sub(replace_match, template)

    # Catch malformed/unresolved template markers.
    if "{{" in rendered or "}}" in rendered:
        return {
            "status": "blocked",
            "rendered_response": None,
            "missing_fields": [],
            "reason": "Unresolved or malformed placeholder remains",
        }

    return {
        "status": "rendered_for_review",
        "rendered_response": rendered,
        "missing_fields": [],
        "reason": "Template values supplied; human approval still required",
    }


# Safety test: deliberately provide no context.
# The expected result is blocked, not a fabricated response.
sample_template = (
    "I can help with order {{Order Number}}. "
    "Please visit {{Website URL}}."
)

test_result = render_response_template(
    sample_template,
    context={},
)

print("Status:", test_result["status"])
print("Missing fields:", test_result["missing_fields"])
print("Rendered response:", test_result["rendered_response"])
print("Reason:", test_result["reason"])

assert test_result["status"] == "blocked"
assert test_result["rendered_response"] is None

print("\nSafety test passed.")

Status: blocked
Missing fields: ['Order Number', 'Website URL']
Rendered response: None
Reason: Required placeholder values are missing

Safety test passed.
